In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "arcturus_back_in_time.gif"

FPS = 24
DURATION_SEC = 12
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#020510"
GRID = "#2b4358"
TEXT = "#c9d3df"
MUTED = "#6f8194"

ARCTURUS = "#ffb35c"
CYAN = "#35c9ff"
ORANGE = "#ff9a3c"
WHITE = "#f0f6ff"
GREEN = "#48ffb3"

rng = np.random.default_rng(42)

# =========================
# Arcturus motion
# =========================

PM_RA = -1093.4     # mas/yr
PM_DEC = -1999.4    # mas/yr

EPOCH_NOW = 2025
EPOCH_PTOLEMY = 150
EPOCH_HIPPARCHUS = -130

def pos(epoch):
    dt = epoch - EPOCH_NOW
    x = PM_RA * dt / 60000.0
    y = PM_DEC * dt / 60000.0
    return x, y

# trajectory
years = np.linspace(EPOCH_HIPPARCHUS, EPOCH_NOW, 1200)
traj_x = np.array([pos(y)[0] for y in years])
traj_y = np.array([pos(y)[1] for y in years])

# =========================
# Figure
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0.06, right=0.98, top=0.90, bottom=0.12)

# auto bounds
ax.set_xlim(traj_x.min()-5, traj_x.max()+5)
ax.set_ylim(traj_y.min()-5, traj_y.max()+5)

ax.grid(True, color=GRID, linestyle=":", alpha=0.5)

for spine in ax.spines.values():
    spine.set_color(GRID)

ax.tick_params(colors=TEXT)

ax.set_xlabel("RA*cos(dec) offset (arcmin)", color=TEXT)
ax.set_ylabel("Dec offset (arcmin)", color=TEXT)

# =========================
# Background stars
# =========================

N = 200
sx = rng.uniform(traj_x.min()-5, traj_x.max()+5, N)
sy = rng.uniform(traj_y.min()-5, traj_y.max()+5, N)

ax.scatter(sx, sy, s=rng.uniform(2,10,N), color=WHITE, alpha=0.4)

# =========================
# Artists
# =========================

line, = ax.plot([], [], color=ARCTURUS, lw=2)
glow, = ax.plot([], [], color=ARCTURUS, lw=8, alpha=0.08)

point = ax.scatter([], [], s=90, color=ARCTURUS, edgecolor=WHITE, zorder=10)

year_text = ax.text(
    0.02, 0.05,
    "",
    transform=ax.transAxes,
    color=ARCTURUS,
    fontsize=16,
    ha="left",
    va="bottom"
)

status = ax.text(
    0.02, 0.12,
    "",
    transform=ax.transAxes,
    color=MUTED,
    fontsize=11
)

# epoch markers
def marker(epoch, color):
    x,y = pos(epoch)
    ax.scatter([x],[y], s=60, color=color, edgecolor=WHITE, zorder=8)
    ax.text(x+1, y+1, str(epoch), color=color, fontsize=10)

marker(EPOCH_NOW, GREEN)
marker(EPOCH_PTOLEMY, ORANGE)
marker(EPOCH_HIPPARCHUS, CYAN)

# =========================
# Helpers
# =========================

def smooth(a,b,x):
    t = np.clip((x-a)/(b-a),0,1)
    return t*t*(3-2*t)

def ease(x):
    return 1-(1-x)**3

# =========================
# Animation
# =========================

def update(frame):
    t = frame/(FRAMES-1)

    # reverse time
    p = ease(smooth(0.05,0.9,t))
    year = EPOCH_NOW + p*(EPOCH_HIPPARCHUS - EPOCH_NOW)

    x,y = pos(year)

    # path
    mask = years >= year
    line.set_data(traj_x[mask], traj_y[mask])
    glow.set_data(traj_x[mask], traj_y[mask])

    point.set_offsets([[x,y]])

    # year label
    if year < 0:
        year_text.set_text(f"{abs(int(year))} BCE")
    else:
        year_text.set_text(f"{int(year)} CE")

    # status
    if year > EPOCH_PTOLEMY:
        status.set_text("moving from modern sky")
        status.set_color(GREEN)
    elif year > EPOCH_HIPPARCHUS:
        status.set_text("passing Ptolemy epoch")
        status.set_color(ORANGE)
    else:
        status.set_text("near Hipparchus epoch")
        status.set_color(CYAN)

    return line, glow, point, year_text, status

# =========================
# Save
# =========================

anim = FuncAnimation(fig, update, frames=FRAMES)

anim.save(GIF_PATH, writer=PillowWriter(fps=FPS))

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print("Saved:", GIF_PATH)